# RAG

In [3]:
import os
import json
import requests
from dotenv import load_dotenv

# 加载.env 文件中的环境变量
load_dotenv()       # 在当前目录寻找 .env 文件

api_key = os.getenv("DEEPSEEK_API_KEY")

if not api_key:
    raise ValueError("请在 .env 文件中设置DEEPSEEK_API_KEY")

# 设置端点url
url = "https://api.deepseek.com/chat/completions"


### 3.2 RAG底层实现

##### 3.2.1 向量化

In [ ]:
from sentence_transformers import SentenceTransformer

# 3.构建请求数据
payload = {
    "model":"deepseek-v4-flash",
    "input":"The food was delicious and the waiter ...",
    "encoding_format":"float"
}

# 4.设置请求头
headers ={
    "Authorization":f"Bearer{api_key}",
    "Content-Type":"application/json"
}

# 5.发送POST请求
response = requests.post(url,headers=headers,json=payload)

# 6.处理响应
if response.status_code == 200:
    result =response.json()
    # 提取嵌入向量（第一个结果）
    embedding = result["data"][0]["embedding"]
    print(f"向量维度：{len(embedding)}")
    print(f"前10个数值：{embedding[:10]}")
    # 打印使用情况
    print(f"Token 消耗：{result['usage']['total_tokens']}")
else:
    print(f"请求失败，状态码：{response.status_code}")
    print(response.text)

In [4]:
from transformers import AutoTokenizer,AutoModel
import torch

# model_path = "AI_Angent/model/Qwen3-Embedding-0.6B-GGUF"  # 可以是相对路径或绝对路径

# # 从与训练模型加载分词器
# tokenizer =AutoTokenizer.from_pretrained(model_path)

# model = AutoModel.from_pretrained(model_path)

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-Embedding-0.6B")
model = AutoModel.from_pretrained("Qwen/Qwen3-Embedding-0.6B")

# 定义需要向量化的文本列表，包括中文和英文
texts =["这是一个测试文本。","This is a test text."]

# 使用分词器对文本进行分词，并设置填充和截断，返回PyTorch张量
inputs = tokenizer(texts,padding=True,truncation=True,return_tensor="pt")

# 禁用梯度计算以节省内存和加快计算
with torch.no_grad():
    embeddings =model(**inputs)     # 通过模型获取文本的向量嵌入

print(embeddings)       # 输出向量嵌入结果

'[Errno 101] Network is unreachable' thrown while requesting HEAD https://huggingface.co/Qwen/Qwen3-Embedding-0.6B/resolve/main/config.json
Retrying in 1s [Retry 1/5].
'[Errno 101] Network is unreachable' thrown while requesting HEAD https://huggingface.co/Qwen/Qwen3-Embedding-0.6B/resolve/main/config.json
Retrying in 1s [Retry 1/5].


OSError: Can't load the configuration of 'Qwen/Qwen3-Embedding-0.6B'. If you were trying to load it from 'https://huggingface.co/models', make sure you don't have a local directory with the same name. Otherwise, make sure 'Qwen/Qwen3-Embedding-0.6B' is the correct path to a directory containing a config.json file

In [2]:
from transformers import AutoTokenizer, AutoModel
import torch
import torch.nn.functional as F

# 1. 加载模型和分词器
tokenizer = AutoTokenizer.from_pretrained("intfloat/multilingual-e5-large")
model = AutoModel.from_pretrained("intfloat/multilingual-e5-large")

# 2. 定义需要向量化的文本
# 注意：根据官方建议，为查询（query）和段落（passage）添加不同前缀会更准确
texts = [
    "query: 这是一个测试文本。", 
    "query: This is a test text."
]

# 3. 定义平均池化函数
def average_pool(last_hidden_states: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
    last_hidden = last_hidden_states.masked_fill(~attention_mask[..., None].bool(), 0.0)
    return last_hidden.sum(dim=1) / attention_mask.sum(dim=1)[..., None]

# 4. 对文本进行分词
batch_dict = tokenizer(texts, max_length=512, padding=True, truncation=True, return_tensors='pt')

# 5. 通过模型获取输出
outputs = model(**batch_dict)

# 6. 执行平均池化，得到句子向量
embeddings = average_pool(outputs.last_hidden_state, batch_dict['attention_mask'])

# 7. (可选) 对向量进行L2归一化，便于计算余弦相似度
embeddings = F.normalize(embeddings, p=2, dim=1)

# 8. 查看结果
print("句子向量的形状:", embeddings.shape)  # 应该是 torch.Size([2, 1024])
print("\n第一个句子的向量（前5个元素）:", embeddings[0][:5])

'[Errno 101] Network is unreachable' thrown while requesting HEAD https://huggingface.co/intfloat/multilingual-e5-large/resolve/main/config.json
Retrying in 1s [Retry 1/5].


KeyboardInterrupt: 

##### 3.2.2 长文本切割

##### 3.2.3 向量存储&相似匹配

In [1]:
import chromadb

# 创建Chroma 客户端实例
chroma_client = chromadb.Client()
# 创建一个名为"my_collection"集合
collection =chroma_client.create_collection(name="my_collection")

# 向集合中添加稳定那个和对应的ID
collection.add(
    documents=[
        "This is a document about pineapple",
        "This is a document about oranges"
    ],
    ids=["id1","id2"]
)

# 查询集合，使用查询文本并获取结果
results = collection.query(
    query_texts=["This is a query document about hawaii"],
    n_results=1
)

print(results)

/home/GUO_Zimeng/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz:   3%|▎         | 2.59M/79.3M [03:44<1:51:03, 12.1kiB/s]


ReadTimeout: The read operation timed out in add.

### 3.3 RAGAS召回评估

### 3.4 自我矫正：CRAG